**KEY PERFORMANCE INDICATORS**

In [0]:
USE CATALOG practice; USE SCHEMA retail;

In [0]:
-- Provide details of completed orders - average order value, total value and count 

WITH order_totals AS (
  SELECT o.order_id,
         SUM(oi.quantity * oi.unit_price) AS order_total
  FROM orders o
  JOIN order_items oi ON o.order_id = oi.order_id
  WHERE o.status = 'completed'
  GROUP BY o.order_id
)
SELECT COUNT(*) AS completed_orders,
       SUM(order_total) AS total_revenue,
       ROUND(AVG(order_total), 2) AS avg_order_value
  FROM order_totals;

completed_orders,total_revenue,avg_order_value
12,1779.74,148.31


**Insight:** There was a total revenue of completed orders of $1,779.74 and an average order value of $148.31 and 12 orders.

In [0]:
--Revenue by product

SELECT p.product_name,
       SUM(oi.quantity) AS units_sold,
       SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items oi
JOIN orders o   ON oi.order_id   = o.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY p.product_name
ORDER BY revenue DESC;

product_name,units_sold,revenue
Stand Mixer,2,499.98
Wireless Earbuds,3,239.97
Mens Running Shoes,3,209.97
Camping Tent (4-person),1,159.99
4K Streaming Stick,3,149.97
Womens Winter Jacket,1,129.99
Yoga Mat,4,119.96
Nonstick Pan Set,1,89.99
Bluetooth Speaker,2,79.98
Mystery Novel: The Quiet Hour,5,74.95


**Insights:**
Stand Mixer is the top earner ($499.98 from just 2 units). The Mystery
Novel sells the most units (5) but earns only $74.95 because of its low price. 
This sales data shows that volume and revenue tell different stories. 
From the analysis, Stand Mixer is the top earner with $499.98 from just 2 units. 
The Mystery Novel sells the most units (5) but earns only $74.95 because of its low price

In [0]:
-- Find the monthly revenue with month-over-month growth

WITH monthly AS (
  SELECT CAST(DATE_TRUNC('MONTH', o.order_date) AS DATE) AS month,
         SUM(oi.quantity * oi.unit_price)                AS revenue
  FROM orders o
  JOIN order_items oi ON o.order_id = oi.order_id
  WHERE o.status = 'completed'
  GROUP BY 1
)
SELECT month,
       revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly
ORDER BY month;

month,revenue,mom_growth_pct
2024-01-01,514.93,null
2024-02-01,384.93,-25.2
2024-03-01,559.90,45.5
2024-04-01,319.98,-42.9


**Insights:** The monthly MOM are Feb -25.2%, Mar +45.5%, Apr -42.9%. 
March has only the peak value of $559.90, and this proves how volatile the revenue is, as just one large changed the month.

In [0]:
-- Revenue by state/location
SELECT c.state,
       SUM(oi.quantity * oi.unit_price) AS revenue
FROM customers c
JOIN orders o       ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id    = oi.order_id
WHERE o.status = 'completed'
GROUP BY c.state
ORDER BY revenue DESC;

state,revenue
CO,549.96
WA,439.92
TX,364.92
IL,354.95
FL,69.99


**Insights:** The highest order value is from Colorado with $549.96, driven by Liam Garcia. On the other hand, Florida is the weakest with a value of $69.99 because Sophia's only order was returned.

In [0]:
-- Customers who have never placed an order 
SELECT c.customer_id,
       CONCAT(c.first_name, ' ', c.last_name) AS full_name,
       c.email,
       c.signup_date
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;

customer_id,full_name,email,signup_date
11,Zoe Patel,zoe.p@email.com,2023-10-12


**Insight:** Zoe Patel is the customer who has never placed an order.

In [0]:
-- Customer segmentation using CASE logic to categorize customers in respect to their lifetime value as high, low, mid or no completed purchase.

WITH customer_value AS (
  SELECT c.customer_id,
         CONCAT(c.first_name, ' ', c.last_name) AS customer,
         COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS lifetime_value
  FROM customers c
  LEFT JOIN orders o
         ON c.customer_id = o.customer_id AND o.status = 'completed'
  LEFT JOIN order_items oi
         ON o.order_id = oi.order_id
  GROUP BY c.customer_id, c.first_name, c.last_name
)
SELECT customer,
       lifetime_value,
       CASE
         WHEN lifetime_value >= 300 THEN 'High'
         WHEN lifetime_value >= 100 THEN 'Mid'
         WHEN lifetime_value >   0  THEN 'Low'
         ELSE 'No completed purchase'
       END AS segment
FROM customer_value
ORDER BY lifetime_value DESC;

customer,lifetime_value,segment
Liam Garcia,459.97,High
Noah Kim,379.96,High
Ava Thompson,364.92,High
Isabella Clark,189.98,Mid
Olivia Martinez,164.97,Mid
Ethan Brown,89.99,Low
James Rodriguez,69.99,Low
Mason Lee,59.96,Low
Sophia Wilson,0.00,No completed purchase
Zoe Patel,0.00,No completed purchase
